In [1]:
import os
import torch
import numpy as np
import h5py
import random

from itertools import product
from tqdm import tqdm
from itertools import chain

from utils import load_model, show_images_grid
from models import VAE_dsprites, VAE_shapes
from datasets import ShapesDataset, DSpritesDataset
from torch.utils.data import DataLoader

In [2]:
RESULTS_PATH = "./results"
DATASET = "3d_shapes" # dsprites / 3d_shapes
VAE_MODE = "beta" # normal / beta / factor

SHOW_IMAGES = False

if DATASET not in ["3d_shapes", "dsprites"]:
    raise ValueError("DATASET must be one of 'dsprites', '3d_shapes'")

if VAE_MODE not in ["normal", "beta", "factor"]:
    raise ValueError("VAE_MODE must be one of 'normal', 'beta', 'factor'")

LOAD_MODEL_DIR = f"{VAE_MODE}_VAE_{DATASET}"
CHAMPION_MODEL_NAME = "champion"

print("---SETTINGS---")
print(f"dataset: {DATASET}")
print(f"vae mode: {VAE_MODE}")
print(f"loading directory: {LOAD_MODEL_DIR}")

---SETTINGS---
dataset: 3d_shapes
vae mode: beta
loading directory: beta_VAE_3d_shapes


In [3]:
LATENT_DIM = 10 if DATASET == "dsprites" else 6

hyperparameters = {
    "device": 'cuda' if torch.cuda.is_available() else 'cpu',
    "train_test_split": 0.8,

    "latent_dim": LATENT_DIM,

    "samples_per_factor": 2000,
    "all_val_batch_size": 512
}

print("\n---HYPERPARAMETERS---")
for key, value in hyperparameters.items():
    print(f"{key}: {value}")


---HYPERPARAMETERS---
device: cuda
train_test_split: 0.8
latent_dim: 6
samples_per_factor: 2000
all_val_batch_size: 512


# Loading Model

In [4]:
path = os.path.join(RESULTS_PATH, LOAD_MODEL_DIR, f"{CHAMPION_MODEL_NAME}.pth")
if DATASET == "dsprites":
    model = load_model(VAE_dsprites(latent_dim=hyperparameters['latent_dim']), path=path, hyperparameters=hyperparameters).to(hyperparameters['device'])
elif DATASET == "3d_shapes":
    model = load_model(VAE_shapes(latent_dim=hyperparameters['latent_dim']), path=path, hyperparameters=hyperparameters).to(hyperparameters['device'])

Model loaded from ./results/beta_VAE_3d_shapes/champion.pth


# Evaluate disentanglement

In [5]:
classifier_examples = [] # to train majority vote classifier

print(f"Loading {DATASET} dataset...")

if DATASET == "dsprites":
    _FACTORS_IN_ORDER = ['shape', 'size', 'rotation', 'x_pos', 'y_pos']
    _GENERATION_FACTOR_IDXS = {'shape': 1, 'size': 2, 'rotation': 3, 'x_pos': 4, 'y_pos': 5}
    _NUM_VALUES_PER_FACTOR = {'shape': 3, 'size': 6, 'rotation': 40, 'x_pos': 32, 'y_pos': 32}

    # Load data
    dataset_zip = np.load('./data/dsprites_64x64.npz')
    images = dataset_zip['imgs']
    latents_values = dataset_zip['latents_values']
    latents_classes = dataset_zip['latents_classes']
    latents_sizes = np.array([1,  3,  6, 40, 32, 32])
    latents_bases = np.concatenate((latents_sizes[::-1].cumprod()[::-1][1:], np.array([1,]))) # [1*32, 1*32*32, ..., 1*32*32*40*6*3] 

    # to get std later on
    all_images_loader = DataLoader(DSpritesDataset(images), batch_size=hyperparameters['all_val_batch_size'])

    # Functions
    def dsprites_sample_latents_batch(batch_size=64):
        '''
        Sample each individual variable (but not the fixed factor) e.g. [[1, 2, 0, 33, 12, 31], ...]
        '''

        samples = np.zeros((batch_size, latents_sizes.size))
        for idx, lat_size in enumerate(latents_sizes):
            if idx == FIXED_FACTOR_IDX:
                samples[:, idx] = GENERATIVE_FACTOR_VALUE
            else:
                samples[:, idx] = np.random.randint(lat_size, size=batch_size)

        return samples
    def dsprites_latent_to_index(latents):
        '''
        Transform samples for each variable into index to get the equivalent image by index: [[1, 2, 0, 33, 12, 31], ...] -> [1263007, ...] 
        '''
        return np.dot(latents, latents_bases).astype(int)

        # Helper function to show images
    def dsprites_sample_latents_all(factor, value, shuffle=True): # currently not used
        '''
        Sample each individual variable e.g. [[1, 2, 0, 33, 12, 31], ...]
        '''

        if factor not in ['shape', 'size', 'rotation', 'x_pos', 'y_pos']:
            raise ValueError("factor must be one of ['shape', 'size', 'rotation', 'x_pos', 'y_pos']")
        
        fixed_idx = _GENERATION_FACTOR_IDXS[factor]
        
        if value >= latents_sizes[fixed_idx] or value < 0: 
            raise ValueError(f"value for {factor} must be between 0 - {latents_sizes[fixed_idx]-1}.")

        value_ranges = [list(range(s)) for s in latents_sizes]
        value_ranges[fixed_idx] = [value] # fix factor with certain value
        all_combinations = np.array(list(product(*value_ranges)))

        if shuffle: 
            all_combinations_shuffled = all_combinations[np.random.permutation(len(all_combinations))]
            return all_combinations_shuffled
            
        return all_combinations

elif DATASET == "3d_shapes":
    _FACTORS_IN_ORDER = ['floor_hue', 'wall_hue', 'object_hue', 'scale', 'shape', 'orientation']
    _GENERATION_FACTOR_IDXS = {'floor_hue': 0, 'wall_hue': 1, 'object_hue': 2, 'scale': 3, 'shape': 4, 'orientation': 5}
    _NUM_VALUES_PER_FACTOR = {'floor_hue': 10, 'wall_hue': 10, 'object_hue': 10, 'scale': 8, 'shape': 4, 'orientation': 15}

    # Load data
    shapes_dataset = h5py.File('./data/3d_shapes.h5', 'r')
    images = np.array(shapes_dataset['images'] ) # array shape [480000,64,64,3], uint8 in range(256)

    # to get std later on
    all_images_loader = DataLoader(ShapesDataset(images), batch_size=hyperparameters['all_val_batch_size'])

    # Functions
    def shapes_get_index(factors):
        """ Converts factors to indices in range(num_data)
        Args:
            factors: np array shape [6,batch_size].
                    factors[i]=factors[i,:] takes integer values in 
                    range(_NUM_VALUES_PER_FACTOR[_FACTORS_IN_ORDER[i]]).

        Returns:
            indices: np array shape [batch_size].
        """
        indices = 0
        base = 1
        for factor, name in reversed(list(enumerate(_FACTORS_IN_ORDER))):
            indices += factors[factor] * base
            base *= _NUM_VALUES_PER_FACTOR[name]
        return indices
    def shapes_sample_imgs_batch(batch_size=64):
        """ Samples a batch of images with fixed_factor=fixed_factor_value, but with
            the other factors varying randomly.
        Args:
            batch_size: number of images to sample.

        Returns:
            batch: images shape [batch_size,64,64,3]
        """
        factors = np.zeros([len(_FACTORS_IN_ORDER), batch_size],
                            dtype=np.int32)
        for factor, name in enumerate(_FACTORS_IN_ORDER):
            num_choices = _NUM_VALUES_PER_FACTOR[name]
            factors[factor] = np.random.choice(num_choices, batch_size)
        
        factors[FIXED_FACTOR_IDX] = GENERATIVE_FACTOR_VALUE
        indices = shapes_get_index(factors)
        
        imgs = []

        for ind in indices:
            im = images[ind]
            im = np.asarray(im)
            imgs.append(im)

        imgs = np.stack(imgs, axis=0)
        imgs = imgs / 255. # normalise values to range [0,1]
        imgs = imgs.astype(np.float32)
        return imgs.reshape([batch_size, 64, 64, 3])

else:
    raise ValueError("DATASET must be one of 'dsprites', '3d_shapes'")


# ------------------------------------------------------------------------------------------------------------------------------------------------
# STEP 0: Get standard deviation for latents of entire dataset 
# ------------------------------------------------------------------------------------------------------------------------------------------------

model.eval()
model.to(hyperparameters['device'])

all_z = []

with torch.no_grad():
    for batch in tqdm(all_images_loader, total=len(all_images_loader), desc="Getting standard deviation for latents of entire dataset"):
        batch = batch.to(hyperparameters['device'])  # (B, 1, 64, 64)
        mu, logvar = model.encode(batch)
        z = model.reparameterize(mu, logvar)  # (B, latent_dim)
        all_z.append(z)

all_z = torch.cat(all_z, dim=0)  # (N, latent_dim)

# get standard deviation for each dimension of latent space
std_all = torch.std(all_z, dim=0, unbiased=False)

print(f"Standard deviation for each dimension of latent space: {std_all}")

# ------------------------------------------------------------------------------------------------------------------------------------------------
# STEP 1: Sample random values for every generative factor
# ------------------------------------------------------------------------------------------------------------------------------------------------

print("Starting to create classifier examples...")

for GENERATIVE_FACTOR in _FACTORS_IN_ORDER:
    FIXED_FACTOR_IDX = _GENERATION_FACTOR_IDXS[GENERATIVE_FACTOR]

    for epoch in tqdm(range(hyperparameters['samples_per_factor']), desc=f"Creating classifier examples for {GENERATIVE_FACTOR}"):
        max_value = _NUM_VALUES_PER_FACTOR[GENERATIVE_FACTOR]-1
        GENERATIVE_FACTOR_VALUE = random.randint(0, max_value)

        # ------------------------------------------------------------------------------------------------------------------------------------------------
        # STEP 2: Get a random batch of images
        # ------------------------------------------------------------------------------------------------------------------------------------------------

        if DATASET == "dsprites":
            # get batch of random samples with fixed factor
            fixed_factor_latents = dsprites_sample_latents_batch()
            fixed_factor_indices = dsprites_latent_to_index(fixed_factor_latents)
            fixed_factor_imgs = np.array(images[fixed_factor_indices])
            
            show_images_grid(fixed_factor_imgs) if SHOW_IMAGES else None

            # handle images as torch tensors with correct shape for encoder
            fixed_factor_imgs_tensor = torch.from_numpy(fixed_factor_imgs).float()  # to float32
            fixed_factor_imgs_tensor = fixed_factor_imgs_tensor.unsqueeze(1)  # (N, 1, 64, 64) → add channel dim

        elif DATASET == "3d_shapes":
            # get batch of random samples with fixed factor
            fixed_factor_img_batch = shapes_sample_imgs_batch()
            
            show_images_grid(fixed_factor_img_batch) if SHOW_IMAGES else None

            # handle images as torch tensors with correct shape for encoder
            fixed_factor_imgs_tensor = torch.from_numpy(fixed_factor_img_batch).float() # to float32
            fixed_factor_imgs_tensor = fixed_factor_imgs_tensor.permute(0, 3, 1, 2) # (N, 3, 64, 64) → create channel first tensor

        else:
            raise ValueError("DATASET must be one of 'dsprites', '3d_shapes'")


        # ------------------------------------------------------------------------------------------------------------------------------------------------
        # STEP 3: Turn images into latents
        # ------------------------------------------------------------------------------------------------------------------------------------------------
        fixed_factor_imgs_tensor = fixed_factor_imgs_tensor.to(hyperparameters['device'])
        mu, logvar = model.encode(fixed_factor_imgs_tensor)
        z = model.reparameterize(mu, logvar)

        z_norm = z / (std_all + torch.tensor(1e-8).to(hyperparameters['device'])) # normalize latents
        var_per_dim = torch.var(z_norm, dim=0, unbiased=False) # get variance for each dimension
        predicted_dim = torch.argmin(var_per_dim).item() # get index of dimension with least variance -> prediction

        classifier_examples.append((predicted_dim, FIXED_FACTOR_IDX))

Loading 3d_shapes dataset...


Getting standard deviation for latents of entire dataset: 100%|██████████| 938/938 [00:36<00:00, 25.75it/s]


Standard deviation for each dimension of latent space: tensor([1.0093, 1.0148, 0.9994, 1.0016, 1.0243, 0.9958], device='cuda:0')
Starting to create classifier examples...


Creating classifier examples for orientation: 100%|██████████| 2000/2000 [00:12<00:00, 163.23it/s]


### Evaluate disentanglement (majority vote)

In [6]:
from collections import defaultdict, Counter

def majority_vote_classifier(train_examples, test_examples):
    """
    Args:
        train_examples: List of (predicted_dim, true_factor_index) from training set
        test_examples: List of (predicted_dim, true_factor_index) from test set

    Returns:
        error_rate: float - lower is better (perfect disentanglement → 0)
    """

    # Count, which true_factor_index occurs most per predicted_dim
    votes = defaultdict(list)
    for pred_dim, true_factor in train_examples:
        votes[pred_dim].append(true_factor)

    # Get most common class for each predicted_dim
    majority_vote = {dim: Counter(factors).most_common(1)[0][0] for dim, factors in votes.items()}

    # Compare to test labels
    correct = 0
    for pred_dim, true_factor in test_examples:
        predicted_factor = majority_vote.get(pred_dim, -1)
        if predicted_factor == true_factor:
            correct += 1

    total = len(test_examples)
    accuracy = correct / total
    error_rate = 1 - accuracy
    return error_rate

In [7]:
split_idx = int(hyperparameters['samples_per_factor'] * hyperparameters['train_test_split'])
examples_per_factor = np.split(np.array(classifier_examples), len(_FACTORS_IN_ORDER))

# turn into flat list of tuples
train_examples = list(chain.from_iterable([examples[:split_idx] for examples in examples_per_factor])) 
test_examples = list(chain.from_iterable([examples[split_idx:] for examples in examples_per_factor]))

In [8]:
err = majority_vote_classifier(train_examples, test_examples)
print(f"Disentanglement score (error rate): {err:.4f}")

Disentanglement score (error rate): 0.0000
